<a href="https://colab.research.google.com/github/lollopelle01/aac-mcp-agent/blob/main/eval/cpu-colab/explore/sequential_blocks/eval_cpu_colab_explore_sb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AACAgent: CPU Evaluation Notebook (llama.cpp / Colab)

Evaluation on standard CPU using `LlamaCppBackend` with Q4_K_M GGUF models.
Designed to run on **Google Colab with CPU runtime** (no GPU required),
to provide a uniform and reproducible hardware baseline.

**NOTE**: the backend is identical to the one used by the production app (`api/server.py`).

**Quick instructions:**
1. On Colab: Runtime → Change runtime type → **CPU**
2. Edit the `colab-env` cell with the desired parameters
3. Run all cells in order

## Table of contents

- [1. Repository setup (Colab only)](#1-repository-setup-colab-only)
- [2. Configuration](#2-configuration)
  - [2.1 Environment variables](#21-environment-variables)
  - [2.2 Resolved parameters](#22-resolved-parameters)
- [3. Environment setup](#3-environment-setup)
  - [3.1 Dependencies installation](#31-dependencies-installation)
  - [3.2 GGUF models download](#32-gguf-models-download)
  - [3.3 Project imports & path setup](#33-project-imports-path-setup)
  - [3.4 Logging setup](#34-logging-setup)
- [4. Dataset preparation](#4-dataset-preparation)
  - [4.1 CSV output schema](#41-csv-output-schema)
  - [4.2 Load annotated dataset](#42-load-annotated-dataset)
  - [4.3 Sample rows for evaluation](#43-sample-rows-for-evaluation)
- [5. Evaluation helpers](#5-evaluation-helpers)
  - [5.1 Gold metadata & teacher forcing](#51-gold-metadata-teacher-forcing)
  - [5.2 Multi-turn execution logic](#52-multi-turn-execution-logic)
- [6. Run evaluation](#6-run-evaluation)

## 1. Repository setup (Colab only)

Clones the repository fresh into `/content/aac-mcp-agent`, removing any stale copy
from a previous run. This ensures the notebook always runs against the latest code
on `main`, matching production behavior.

In [1]:
##### COLAB ONLY, here we clone the repo ##################################################
import subprocess, sys, os
import shutil

PROJECT_ROOT = "/content/aac-mcp-agent"

# Ensure we are in a known good directory before doing anything
os.chdir("/content/")

# Remove existing directory if it exists
# NOTE: useful when testing under multipel pushes
if os.path.exists(PROJECT_ROOT):
    print(f"Removing existing directory: {PROJECT_ROOT}")
    shutil.rmtree(PROJECT_ROOT)

try:
    result = subprocess.run(
        ["git", "clone", "https://github.com/lollopelle01/aac-mcp-agent.git",
         PROJECT_ROOT],
        check=True,
        capture_output=True,
        text=True
    )
    print(result.stdout)
except subprocess.CalledProcessError as e:
    print(f"Error cloning repository: {e.cmd}")
    print(f"Return code: {e.returncode}")
    print(f"Output (stdout): {e.stdout}")
    print(f"Error (stderr): {e.stderr}")
    sys.exit(1)

# Restore original behavior of changing into PROJECT_ROOT *after* cloning
os.chdir(PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, "app", "src"))
sys.path.insert(0, os.path.join(PROJECT_ROOT, "app"))

print("Repo cloned. CWD:", os.getcwd())


Repo cloned. CWD: /content/aac-mcp-agent


## 2. Configuration

### 2.1 Environment variables

Edit the values below before running. These control which models are evaluated,
how many sentences are sampled, and where outputs are written.

> ⚠️ **Path note**: `NB_OUTPUT_CSV` must point inside this notebook's own folder
> (`eval/cpu-colab/explore/round_robin_weighted/`) so that `metrics_eval_cpu_rrw.ipynb`
> can find it afterwards.

In [2]:
import os

# ENV VARS: edit here before running #####################################
os.environ["NB_MODELS"]            = "qwen2.5:3b llama3.2:3b granite4:3b-h"
os.environ["NB_N_ROWS"]            = "30"                                   # 0 = all sentences
os.environ["NB_LANG"]              = "en_eval"
os.environ["NB_MAX_RESULTS"]       = "0"                                    # 0 = use default from app/settings.py
os.environ["NB_SEED"]              = "42"
os.environ["NB_SPLIT_FILTER"]      = "all"                                  # "clear" | "vague" | "all"
os.environ["NB_N_THREADS"]         = "2"                                    # NOTE: Colab CPU has 2 vCPUs
os.environ["NB_N_CTX"]             = "2048"
os.environ["NB_OUTPUT_CSV"]        = "/content/aac-mcp-agent/eval/cpu-colab/explore/sequential_blocks/eval_cpu_colab.csv"
os.environ["NB_ANNOTATED_PARQUET"] = "/content/aac-mcp-agent/annotation/eval_final.parquet"

### 2.2 Resolved parameters

Reads back the environment variables (with sensible fallbacks) and prints a summary
before starting any heavy computation.

In [3]:
# Connecting env vars + backup if not selected

MODELS_RAW        = os.environ.get("NB_MODELS",               "qwen2.5:3b")
N_ROWS_ENV        = os.environ.get("NB_N_ROWS",               "100")
LANG_CODE         = os.environ.get("NB_LANG",                 "en_eval")
_max_results_env  = int(os.environ.get("NB_MAX_RESULTS",      "0"))
SEED              = int(os.environ.get("NB_SEED",             "42"))
SPLIT_FILTER      = os.environ.get("NB_SPLIT_FILTER",         "all")
N_THREADS         = int(os.environ.get("NB_N_THREADS",        "2"))
N_CTX             = int(os.environ.get("NB_N_CTX",            "512"))
ANNOTATED_PARQUET = os.environ.get("NB_ANNOTATED_PARQUET",    "/content/aac-mcp-agent/annotation/eval_final.parquet")
OUTPUT_CSV        = os.environ.get("NB_OUTPUT_CSV",           "/content/aac-mcp-agent/eval/cpu-colab/explore/sequential_blocks/eval_cpu_colab.csv")

MODELS = MODELS_RAW.split()
N_ROWS = int(N_ROWS_ENV)

print(f"Models            : {MODELS}")
print(f"N_rows            : {N_ROWS if N_ROWS > 0 else 'full dataset (1760)'}")
print(f"Seed              : {SEED}")
print(f"Lang              : {LANG_CODE}")
print(f"Max results       : {_max_results_env if _max_results_env > 0 else 'default (settings.py)'}")
print(f"Split filter      : {SPLIT_FILTER}")
print(f"n_threads (Colab) : {N_THREADS}")
print(f"n_ctx             : {N_CTX}")
print(f"Annotated parquet : {ANNOTATED_PARQUET}")
print(f"Output CSV        : {OUTPUT_CSV}")

Models            : ['qwen2.5:3b', 'llama3.2:3b', 'granite4:3b-h']
N_rows            : 30
Seed              : 42
Lang              : en_eval
Max results       : default (settings.py)
Split filter      : all
n_threads (Colab) : 2
n_ctx             : 2048
Annotated parquet : /content/aac-mcp-agent/annotation/eval_final.parquet
Output CSV        : /content/aac-mcp-agent/eval/cpu-colab/explore/sequential_blocks/eval_cpu_colab.csv


## 3. Environment setup

### 3.1 Dependencies installation

Installs the lightweight Python dependencies first, then attempts to install
`llama-cpp-python` from a precompiled CPU wheel (avoids a slow from-source build).

In [4]:
import subprocess, sys, importlib

def _pip(*args, **kwargs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

### 1. Lightweight dependencies ################################################################################
_pip(
    "pandas>=2.0", "pyarrow>=14", "tqdm>=4.66",
    "spacy>=3.7", "fastmcp", "pydantic>=2.0",
    "httpx>=0.24", "python-dotenv",
    "sentence-transformers"
)

### 2. llama-cpp-python, precompiled wheel (CPU) ################################################################
def _llama_installed():
    try:
        importlib.import_module("llama_cpp")
        return True
    except ImportError:
        return False

if _llama_installed():
    print("llama-cpp-python is already installed, skipping.")
else:
    import platform, sys as _sys

    # Detect Python version to select the correct wheel
    vi = _sys.version_info
    py_tag = f"cp{vi.major}{vi.minor}"          # e.g. cp311

    # CPU-only precompiled wheel URL from the official abetlen/llama-cpp-python repository
    LLAMA_VERSION = "0.3.4"
    WHEEL_BASE = (
        "https://github.com/abetlen/llama-cpp-python/releases/download"
        f"/v{LLAMA_VERSION}"
    )

    # Linux x86_64 CPU wheel covers Colab
    wheel_url = (
        f"{WHEEL_BASE}/llama_cpp_python-{LLAMA_VERSION}"
        f"-{py_tag}-{py_tag}-linux_x86_64.whl"
    )

    print(f"Attempting precompiled wheel installation: {wheel_url}")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", wheel_url],
        capture_output=True,
    )

    if result.returncode == 0 and _llama_installed():
        print("llama-cpp-python installed from precompiled wheel (~30 s).")
    else:
        # Fallback: official wheel index (still avoids compilation)
        print("Direct wheel not found -> falling back to precompiled index...")
        _pip(
            "llama-cpp-python",
            "--extra-index-url",
            "https://abetlen.github.io/llama-cpp-python/whl/cpu",
        )
        print("llama-cpp-python installed from CPU wheel index.")

### 3. spaCy model ################################################################################
try:
    import spacy
    spacy.load("en_core_web_sm")
    print("spaCy en_core_web_sm is already available.")
except OSError:
    subprocess.check_call(
        [sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"]
    )
    print("spaCy en_core_web_sm downloaded.")

Attempting precompiled wheel installation: https://github.com/abetlen/llama-cpp-python/releases/download/v0.3.4/llama_cpp_python-0.3.4-cp312-cp312-linux_x86_64.whl
Direct wheel not found -> falling back to precompiled index...
llama-cpp-python installed from CPU wheel index.
spaCy en_core_web_sm is already available.


### 3.2 GGUF models download

GGUF weight files are too large to keep in the repo, so they are downloaded here
directly from HuggingFace Hub into `app/models/`. The alias → (repo_id, filename)
mapping is duplicated here (rather than imported from `settings.py`) because on
Colab the project path isn't importable yet at this point.

In [5]:
from pathlib import Path

#### Download GGUFs from HuggingFace #########################################

# 1) GGUF files are not in the repo as they are too large, so we need to download
# them here directly from HuggingFace Hub and saved to app/models/.

# 2) The alias -> (repo_id, filename) mapping is defined here because on Colab
# we cannot read settings.py before the path setup.

_MODELS_DIR = Path("/content/aac-mcp-agent/app/models")
_MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Full mapping (update if new models are added to settings.py)
_GGUF_SOURCES = {
    "qwen2.5:3b":    ("bartowski/Qwen2.5-3B-Instruct-GGUF",         "Qwen2.5-3B-Instruct-Q4_K_M.gguf"),
    "llama3.2:3b":   ("bartowski/Llama-3.2-3B-Instruct-GGUF",       "Llama-3.2-3B-Instruct-Q4_K_M.gguf"),
    "granite4:3b-h": ("bartowski/ibm-granite_granite-4.1-3b-GGUF",   "ibm-granite_granite-4.1-3b-Q4_K_M.gguf"),
    "mistral:7b":    ("bartowski/Mistral-7B-Instruct-v0.3-GGUF",     "Mistral-7B-Instruct-v0.3-Q4_K_M.gguf"),
}

_pip("huggingface_hub>=0.22")
from huggingface_hub import hf_hub_download

downloaded: dict[str, str] = {}  # alias -> absolute path

for alias in MODELS:
    if alias not in _GGUF_SOURCES:
        print(f"'{alias}' has no GGUF source defined in _GGUF_SOURCES")
        continue
    repo_id, filename = _GGUF_SOURCES[alias]
    dest = _MODELS_DIR / filename
    if dest.exists():
        print(f"{alias}: already present ({dest.name})")
        downloaded[alias] = str(dest)
        continue
    print(f"{alias}: downloading {filename} from {repo_id} ...")
    path = hf_hub_download(
        repo_id   = repo_id,
        filename  = filename,
        local_dir = str(_MODELS_DIR),
    )
    downloaded[alias] = path
    print(f"saved to {path}")

missing = [m for m in MODELS if m not in downloaded]
if missing:
    raise RuntimeError(f"Missing GGUFs for: {missing}. Add them to _GGUF_SOURCES.")

qwen2.5:3b: downloading Qwen2.5-3B-Instruct-Q4_K_M.gguf from bartowski/Qwen2.5-3B-Instruct-GGUF ...


Qwen2.5-3B-Instruct-Q4_K_M.gguf:   0%|          | 0.00/1.93G [00:00<?, ?B/s]

saved to /content/aac-mcp-agent/app/models/Qwen2.5-3B-Instruct-Q4_K_M.gguf
llama3.2:3b: downloading Llama-3.2-3B-Instruct-Q4_K_M.gguf from bartowski/Llama-3.2-3B-Instruct-GGUF ...


Llama-3.2-3B-Instruct-Q4_K_M.gguf:   0%|          | 0.00/2.02G [00:00<?, ?B/s]

saved to /content/aac-mcp-agent/app/models/Llama-3.2-3B-Instruct-Q4_K_M.gguf
granite4:3b-h: downloading ibm-granite_granite-4.1-3b-Q4_K_M.gguf from bartowski/ibm-granite_granite-4.1-3b-GGUF ...


ibm-granite_granite-4.1-3b-Q4_K_M.gguf:   0%|          | 0.00/2.17G [00:00<?, ?B/s]

saved to /content/aac-mcp-agent/app/models/ibm-granite_granite-4.1-3b-Q4_K_M.gguf


### 3.3 Project imports & path setup

Adds `app/` and `app/src/` to `sys.path` so the notebook can import the same
agent code used in production.

In [6]:
PROJECT_ROOT = Path("/content/aac-mcp-agent")
APP          = PROJECT_ROOT / "app"
SRC          = APP / "src"

for p in [str(SRC), str(APP)]:
    if p not in sys.path:
        sys.path.insert(0, p)

EVAL_PARQUET = Path(ANNOTATED_PARQUET)

_out = Path(OUTPUT_CSV)
_out.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV_PATH = _out

print(f"PROJECT_ROOT  : {PROJECT_ROOT}")
print(f"Eval parquet  : {EVAL_PARQUET}  exists={EVAL_PARQUET.exists()}")
print(f"Output CSV    : {OUTPUT_CSV_PATH}")

PROJECT_ROOT  : /content/aac-mcp-agent
Eval parquet  : /content/aac-mcp-agent/annotation/eval_final.parquet  exists=True
Output CSV    : /content/aac-mcp-agent/eval/cpu-colab/explore/sequential_blocks/eval_cpu_colab.csv


### 3.4 Logging setup

Routes the agent's internal `agent.run` logger to a file (`agent_run.log`) instead
of stdout, to keep the notebook output clean while still preserving full
plan/context/resolve traces for later debugging.

In [7]:
import ast
import csv
import logging
import time
from pathlib import Path

import pandas as pd

# Silence noisy libs (keep WARNING+ only)
logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
for _noisy in ("httpx", "urllib3", "llama_cpp"):
    logging.getLogger(_noisy).setLevel(logging.WARNING)

# Project imports
from config import AGENT_MAX_RESULTS
from settings import settings
from agent.agent import AACAgent, EvalContext
from agent.backends import LlamaCppBackend
from agent.session import SessionMemory
from mcp_server.models import Pictogram, Keyword
from mcp_server.tools.arasaac import get_pictogram_metadata
import mcp_server.tools.arasaac as _arasaac_mod

_arasaac_mod.LANG = LANG_CODE

EVAL_MAX_RESULTS = _max_results_env if _max_results_env > 0 else AGENT_MAX_RESULTS

# Route agent.run to file (captures [PLAN OUT], [PLAN], [CTX], [EVAL], [RESOLVE], [FALLBACK])
# setup_logging() is NOT called: it uses relative paths that don't exist on Colab, all would fall in /app
_log_path = Path(OUTPUT_CSV_PATH).parent / "agent_run.log"
_fh = logging.FileHandler(_log_path, mode="w", encoding="utf-8")
_fh.setLevel(logging.INFO)
_fh.setFormatter(logging.Formatter("%(message)s"))
_agent_log = logging.getLogger("agent.run")
_agent_log.handlers.clear()   # remove stale handlers from previous runs
_agent_log.addHandler(_fh)
_agent_log.setLevel(logging.INFO)
_agent_log.propagate = False  # don't fall through to basicConfig (WARNING)

_log_drive_dst: Path | None = None   # will be set by IncrementalCSV.__init__

print(f"EVAL_MAX_RESULTS : {EVAL_MAX_RESULTS}")
print(f"agent.run log    : {_log_path}")

EVAL_MAX_RESULTS : 50
agent.run log    : /content/aac-mcp-agent/eval/cpu-colab/explore/sequential_blocks/agent_run.log


## 4. Dataset preparation

### 4.1 CSV output schema

Defines the columns of the incremental output CSV, one row per turn.

In [8]:
import time

# CSV columns
CSV_COLUMNS = [
    "row_idx",
    "input_type",
    "turn_pos",
    "concept_text",
    "called_get_time",
    "called_get_schedule",
    "needs_context",
    "predicted_ids",
    "pool_ids",
    "plan_method",
    "resolve_method",
    "planner_concepts",
    "turn_time_s",
    "model_name",
    "window_size",
]

### 4.2 Load annotated dataset

Loads the annotated evaluation parquet (`eval_final.parquet`) and normalizes the
`concepts`/`schedule` columns back into Python lists (parquet round-trips them
as arrays/strings depending on the writer).

In [9]:
import numpy as np

df_full = pd.read_parquet(EVAL_PARQUET)

# Parquet serializes list not as python lists, so we convert them
def _to_list(x):
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, str):
        return ast.literal_eval(x)
    if isinstance(x, list):
        return x
    return []

df_full["concepts"] = df_full["concepts"].apply(_to_list)
df_full["schedule"] = df_full["schedule"].apply(_to_list)

print(f"Dataset: {len(df_full):,} rows  x  {list(df_full.columns)} columns")
n_with_sched = (df_full["schedule"].apply(len) > 0).sum()
print(f"Rows with schedule events: {n_with_sched:,} / {len(df_full):,}")

Dataset: 1,760 rows  x  ['sentence', 'concepts', 'caregiver_clear', 'caregiver_vague', 'time_of_day', 'event_time', 'schedule', 'tod_selection', 'split'] columns
Rows with schedule events: 1,760 / 1,760


### 4.3 Sample rows for evaluation

Draws a reproducible random sample of `N_ROWS` sentences (capped at 200, since a
larger sample is not feasible on Colab's CPU runtime within a reasonable time budget).

In [10]:
_cap = N_ROWS if N_ROWS > 0 else 200   # more than 200 rows is infiseable on colab

df = (
    df_full
    .sample(min(_cap, len(df_full)), random_state=SEED)
    .reset_index()
    .rename(columns={"index": "row_idx"})
    .reset_index(drop=True)
)

print(f"Sample: {len(df)} rows (cap={_cap}, seed={SEED})")
print(f"row_idx range: {df['row_idx'].min()} - {df['row_idx'].max()}")

Sample: 30 rows (cap=30, seed=42)
row_idx range: 70 - 1683


## 5. Evaluation helpers

### 5.1 Gold metadata & teacher forcing

Helpers to fetch gold pictogram metadata, build the mock `EvalContext` for each
turn (time/schedule), and inject the gold pictogram back into agent memory after
each turn (teacher forcing), so errors don't compound across turns.

In [11]:
### Helper: gold metadata ################################################################

_gold_cache: dict[int, dict] = {}

def get_gold_meta(pic_id: int) -> dict:
    k = int(pic_id)
    if k not in _gold_cache:
        try:
            _gold_cache[k] = get_pictogram_metadata(pictogram_id=k, lang=LANG_CODE)
        except Exception:
            _gold_cache[k] = {}
    return _gold_cache[k]

def gold_as_pictogram(pic_id: int, concept: str) -> Pictogram:
    """Build a Pictogram for teacher forcing using the dataset concept_text as keyword."""
    return Pictogram(id=int(pic_id), keywords=[Keyword(type=2, keyword=concept)])

def build_eval_ctx(row, turn_pos: int) -> EvalContext:
    """Build the EvalContext for a given turn."""
    from datetime import datetime, date, time as dtime
    event_time_str = str(row["event_time"])
    try:
        h, m = map(int, event_time_str.split(":"))
        current_dt = datetime.combine(date.today(), dtime(h, m)).isoformat()
    except Exception:
        current_dt = datetime.now().isoformat()

    mock_time = {
        "current_dt":  current_dt,
        "time_of_day": row["time_of_day"],
    }

    raw_sched = row["schedule"]
    if isinstance(raw_sched, np.ndarray):
        raw_sched = raw_sched.tolist()
    elif isinstance(raw_sched, str):
        raw_sched = ast.literal_eval(raw_sched)
    mock_schedule = raw_sched if isinstance(raw_sched, list) else []

    return EvalContext(
        mock_time=mock_time,
        mock_schedule=mock_schedule,
        mock_needs_context=None if turn_pos==0 else False,   # let the model decide for real at turn 0, do not in turns 1+
    )

def teacher_force(agent: AACAgent, gold_id: int, concept: str) -> None:
    """Inject the gold pictogram into memory after each turn."""
    if not agent.memory.turns:
        return
    last = agent.memory.turns[-1]
    gold_pic = gold_as_pictogram(gold_id, concept)
    last.pictograms = [gold_pic]


### Helper ############################################################################################
# NOTE: this is very useful if the runtime disconnects and to check the run remotely
class IncrementalCSV:
    def __init__(self, path: Path, drive_backup: bool = True) -> None:
        self.path       = path
        self._is_new    = not path.exists()
        self._buffer:   list[dict] = []
        self._drive_dst: Path | None = None

        if drive_backup:
            try:
                from google.colab import drive as _drive
                import shutil as _shutil
                self._shutil = _shutil
                _drive.mount("/content/drive", force_remount=False)
                _backup_dir = Path("/content/drive/MyDrive/aac_eval")
                _backup_dir.mkdir(parents=True, exist_ok=True)
                self._drive_dst = _backup_dir / path.name
                # Register log backup path on the module-level variable set in cell 8
                global _log_drive_dst
                _log_drive_dst = _backup_dir / _log_path.name
                print(f"[Drive] mounted ==> backup CSV : {self._drive_dst}")
                print(f"[Drive]            backup log  : {_log_drive_dst}")
            except Exception as e:
                print(f"[Drive] mount failed (skip backup): {e}")

    def add(self, rows: list[dict]) -> None:
        self._buffer.extend(rows)

    def flush(self) -> None:
        if not self._buffer:
            return
        mode = "w" if self._is_new else "a"
        with open(self.path, mode, newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS, extrasaction="ignore")
            if self._is_new:
                writer.writeheader()
                self._is_new = False
            writer.writerows(self._buffer)
        self._buffer.clear()

        if self._drive_dst is not None:
            try:
                self._shutil.copy2(self.path, self._drive_dst)
            except Exception as e:
                print(f"[Drive] failed CSV backup: {e}")

        if _log_drive_dst is not None:
            try:
                _fh.flush()
                self._shutil.copy2(_log_path, _log_drive_dst)
            except Exception as e:
                print(f"[Drive] failed log backup: {e}")

### 5.2 Multi-turn execution logic

Defines `run_multi_turn`, which replays a full sentence turn-by-turn:
- **Turn 0**: real caregiver input (clear/vague) + full mock context; the model
  decides for itself whether context is needed.
- **Turn 1+**: empty input, context already resolved via session history.
- After each turn, the gold pictogram is force-injected into memory.

In [12]:
def run_multi_turn(agent: AACAgent, row: "pd.Series", input_type: str, model_name: str, window_size: int) -> list[dict]:
    """Run the multi-turn sequence for a row and an input_type.

    - Turn 0: real input (clear or vague) + FULL EvalContext with mock tools;
              the decision LLM decides for real whether context is needed
              (see build_eval_ctx).
    - Turn 1+: input="" + mock_needs_context=False (context already in session history).
    - Teacher forcing: after each turn the gold is injected into memory.
    - Turns are 0-based throughout (turn 0 is the first turn), consistent
      with agent.py's Turn.turn_id numbering.
    """
    concepts = row["concepts"]
    caregiver_input_t0 = str(row["caregiver_clear" if input_type == "clear" else "caregiver_vague"])

    agent.reset_session()
    results: list[dict] = []

    # Sentence separation
    _agent_log.info(
        "\n" + "#" * 300 + "\n" +
        "#" * 10 + " sentence row=%s input_type=%s event_time=%s time_of_day=%s\n" +
        "#" * 10 + " text=%r\n" +
        "#" * 300,
        row["row_idx"], input_type, row["event_time"], row["time_of_day"], caregiver_input_t0,
    )

    for turn_pos, concept_entry in enumerate(concepts):
        concept_text = concept_entry["concept_text"]
        gold_id      = int(concept_entry["gold_id"])

        # Turn of sentence separator
        _turn_sep = f"----- turn={turn_pos} concept={concept_text!r}"
        _agent_log.info("%s", _turn_sep + "-" * max(0, 300 - len(_turn_sep)))

        ec        = build_eval_ctx(row, turn_pos)
        raw_input = caregiver_input_t0 if turn_pos == 0 else ""

        start_time = time.monotonic()
        window = agent.run(raw_input, eval_ctx=ec)
        end_time = time.monotonic()

        predicted_ids    = [p.id for p in window]
        pool_ids         = list(agent.last_pool_ids)
        planner_concepts = [e["concept"] for e in agent.last_resolve_info]
        resolve_method   = [e["method"]  for e in agent.last_resolve_info]

        results.append({
            "row_idx":             row["row_idx"],
            "input_type":          input_type,
            "turn_pos":            turn_pos,
            "concept_text":        concept_text,
            "called_get_time":     "get_time"     in agent.last_tool_calls,
            "called_get_schedule": "get_schedule" in agent.last_tool_calls,
            "needs_context":       agent.last_needs_context,
            "predicted_ids":       str(predicted_ids),
            "pool_ids":            str(pool_ids),
            "plan_method":         agent.last_plan_method,
            "resolve_method":      resolve_method,
            "planner_concepts":    str(planner_concepts),
            "turn_time_s":         round(end_time - start_time, 3),
            "model_name":          model_name,
            "window_size":         window_size,
        })

        teacher_force(agent, gold_id, concept_text)

    return results

## 6. Run evaluation

Main loop: for each window size and model, loads the GGUF backend, instantiates
the agent with the `round_robin_weighted` ranking strategy, and runs every
sampled sentence turn-by-turn, writing results incrementally to
`OUTPUT_CSV_PATH`.

In [13]:
from tqdm.notebook import tqdm

SAVE_EVERY = 1

csv_writer = IncrementalCSV(OUTPUT_CSV_PATH)

WINDOW_SIZES = [10]

for window_size in WINDOW_SIZES:
    print(f"\n{'═'*100}\n  WINDOW SIZE: {window_size}\n{'═'*100}")
    for model_alias in MODELS:
        print(f"\n{'━'*70}\n  MODEL: {model_alias}\n{'━'*70}")

        gguf_path = downloaded.get(model_alias)
        if not gguf_path:
            print(f"   GGUF not available for '{model_alias}', skipping")
            continue

        backend = LlamaCppBackend(
            model_path  = gguf_path,
            n_ctx       = N_CTX,
            n_threads   = N_THREADS,
            temperature = 0.0,
            max_tokens  = 300, # before it was 150 but some inputs require more tokens
            verbose     = False,
        )

        agent = AACAgent(
            model          = model_alias,
            backend        = backend,
            lang           = LANG_CODE,
            max_results    = window_size, # Pass the current window_size here
            fetch_schedule = False,   # mocked via EvalContext, no live calls
            ranking_strategy = "sequential_blocks"
        )

        print("  Loading GGUF into RAM ...", flush=True)
        t_load = time.monotonic()
        agent.backend._ensure_loaded()
        print(f"  GGUF loaded in {time.monotonic() - t_load:.1f}s", flush=True)

        n_errors = 0
        pbar = tqdm(df.iterrows(), total=len(df), desc=f"{model_alias} (window={window_size})")

        for _, row in pbar:
            split_val = str(row["split"])

            if split_val == "none":
                continue
            elif split_val == "clear":
                input_types = ["clear"]
            elif split_val == "vague":
                input_types = ["vague"]
            else:  # "both"
                input_types = ["clear", "vague"]

            if SPLIT_FILTER != "all":
                input_types = [t for t in input_types if t == SPLIT_FILTER]
            if not input_types:
                continue

            try:
                for input_type in input_types:
                    row_results = run_multi_turn(agent, row, input_type, model_alias, window_size)
                    csv_writer.add(row_results)
                if pbar.n % SAVE_EVERY == 0:
                    csv_writer.flush()
            except Exception as exc:
                n_errors += 1
                pbar.set_postfix(errors=n_errors)
                print(f"  [ERROR] row={{row['row_idx']}}: {exc}", flush=True)

        csv_writer.flush()
        agent.unload()
        print(f"  Model {model_alias!r} (window={window_size}) completed.")

print(f"\nDone. Output: {OUTPUT_CSV_PATH}")

Mounted at /content/drive
[Drive] mounted ==> backup CSV : /content/drive/MyDrive/aac_eval/eval_cpu_colab.csv
[Drive]            backup log  : /content/drive/MyDrive/aac_eval/agent_run.log

════════════════════════════════════════════════════════════════════════════════════════════════════
  WINDOW SIZE: 10
════════════════════════════════════════════════════════════════════════════════════════════════════

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  MODEL: qwen2.5:3b
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Loading GGUF into RAM ...
  GGUF loaded in 6.5s


qwen2.5:3b (window=10):   0%|          | 0/30 [00:00<?, ?it/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

  Model 'qwen2.5:3b' (window=10) completed.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  MODEL: llama3.2:3b
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Loading GGUF into RAM ...
  GGUF loaded in 20.8s


llama3.2:3b (window=10):   0%|          | 0/30 [00:00<?, ?it/s]

  Model 'llama3.2:3b' (window=10) completed.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  MODEL: granite4:3b-h
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Loading GGUF into RAM ...
  GGUF loaded in 23.9s


granite4:3b-h (window=10):   0%|          | 0/30 [00:00<?, ?it/s]

  Model 'granite4:3b-h' (window=10) completed.

Done. Output: /content/aac-mcp-agent/eval/cpu-colab/explore/sequential_blocks/eval_cpu_colab.csv
